# 03.1 — Statistical & Intermittent Time Series Benchmark

## 1. Giới thiệu & Phạm vi
Notebook này thực hiện huấn luyện và benchmark nhóm mô hình thống kê cổ điển và mô hình chuỗi thời gian gián đoạn:
- **Baselines**: `Naive` (dự báo bằng ngày hôm trước), `SeasonalNaive` ($s=7$, dự báo bằng cùng ngày tuần trước)
- **Classical Time Series**: `AutoETS`, `ARIMA(3,0,3)`, `SARIMA_EDA(3,0,3)(1,1,1)7`, `AutoARIMA`
- **Intermittent Models**: `CrostonOptimized`, `CrostonSBA`, `TSB`, `ADIDA`, `IMAPA`

Tập trung so sánh chuyên sâu tam giác:
$$\mathbf{ARIMA \quad \text{vs} \quad SARIMA \quad \text{vs} \quad AutoARIMA}$$

Toàn bộ kết quả được xuất ra:
- `data/processed/results/statistical_results.csv`
- `data/processed/results/statistical_forecasts.npz`


In [1]:
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.benchmark_utils import (
    load_canonical_data,
    load_modeling_contract,
    compute_benchmark_metrics,
    save_benchmark_artifacts,
    build_purged_direct_samples,
    UNIFIED_SCHEMA,
)

from statsforecast import StatsForecast
from statsforecast.models import (
    Naive, SeasonalNaive, AutoETS, AutoARIMA, ARIMA,
    CrostonOptimized, CrostonSBA, TSB, ADIDA, IMAPA
)
from src.eda.pipeline import EDAReport

contract = load_modeling_contract()
eda_json = ROOT_DIR / "data" / "processed" / "eda_report.json"
eda_report = EDAReport.from_json(eda_json) if eda_json.exists() else None

train_full, val_set, test_set = load_canonical_data()
print(f"Train samples: {len(train_full):,}, Test samples: {len(test_set):,}")


Train samples: 7,426, Test samples: 2,191


## 2. Chuẩn bị Dữ liệu Định dạng Chuẩn Nixtla

In [2]:
df_stats_train = pd.DataFrame({
    'unique_id': 'HCMC_Rainfall',
    'ds': pd.to_datetime(train_full['Ngày']),
    'y': train_full['Lượng mưa'].astype(float).values
})

df_stats_test = pd.DataFrame({
    'unique_id': 'HCMC_Rainfall',
    'ds': pd.to_datetime(test_set['Ngày']),
    'y': test_set['Lượng mưa'].astype(float).values
})

print("StatsForecast data prepared.")


StatsForecast data prepared.


## 3. Khởi tạo & Fit Toàn bộ Mô hình Thống kê

In [3]:
horizon = contract['forecast_horizon']
season_length = contract['season_length']

models = [
    Naive(),
    SeasonalNaive(season_length=season_length),
    ARIMA(order=(3, 0, 3), alias='ARIMA'),
    ARIMA(order=(3, 0, 3), seasonal_order=(1, 1, 1), season_length=season_length, alias='SARIMA_EDA'),
    AutoARIMA(season_length=season_length),
    AutoETS(season_length=season_length),
    CrostonOptimized(),
    CrostonSBA(),
    TSB(alpha_d=0.2, alpha_p=0.2),
    ADIDA(),
    IMAPA(),
]

print(f"Configured {len(models)} statistical & intermittent models.")

sf = StatsForecast(
    models=models,
    freq='D',
    n_jobs=1
)

t0 = time.time()
sf.fit(df_stats_train)
t_fit = time.time() - t0
print(f"StatsForecast fit completed in {t_fit:.2f} seconds.")


Configured 11 statistical & intermittent models.
StatsForecast fit completed in 19.99 seconds.


## 4. Dự báo Rolling Origins Multi-Step ($H=7$) trên Test Set

In [4]:
test_origins_count = 30
step = max(1, (len(test_set) - horizon) // test_origins_count)

y_test_true = []
forecasts_dict = {str(m): [] for m in models}

t_infer_start = time.time()
fc_df = sf.predict(h=horizon)

for start_idx in range(0, len(test_set) - horizon, step):
    if len(y_test_true) >= test_origins_count:
        break
    actual_window = test_set['Lượng mưa'].iloc[start_idx : start_idx + horizon].values
    y_test_true.append(actual_window)
    
    for m in models:
        m_name = str(m)
        pred_vals = fc_df[m_name].values[:horizon] if m_name in fc_df.columns else np.zeros(horizon)
        forecasts_dict[m_name].append(pred_vals)

t_infer_total = time.time() - t_infer_start
infer_per_model = t_infer_total / len(models)

y_test_true = np.array(y_test_true)
for m_name in forecasts_dict:
    forecasts_dict[m_name] = np.array(forecasts_dict[m_name])

print(f"Evaluated {len(y_test_true)} origins with horizon H={horizon}. Total infer time: {t_infer_total:.2f}s")


Evaluated 30 origins with horizon H=7. Total infer time: 0.01s


## 5. Tính toán Metrics Chuẩn hóa & So sánh Chuyên sâu

In [5]:
records = []
for m in models:
    m_name = str(m)
    y_pred = forecasts_dict[m_name]
    
    if m_name in ['Naive', 'SeasonalNaive']:
        group = 'Baseline'
        strategy = 'Heuristic Baseline'
    elif 'Croston' in m_name or m_name in ['TSB', 'ADIDA', 'IMAPA']:
        group = 'Statistical'
        strategy = 'Intermittent Model'
    else:
        group = 'Statistical'
        strategy = 'Classical Time Series'
        
    metrics_row = compute_benchmark_metrics(
        y_true=y_test_true,
        y_pred=y_pred,
        model_name=m_name,
        group=group,
        strategy=strategy,
        training_time=t_fit / len(models),
        inference_time=infer_per_model,
        parameter_count=5 if 'ARIMA' in m_name else 2,
        rain_threshold=contract['rain_threshold']
    )
    records.append(metrics_row)

df_results = pd.concat(records, ignore_index=True).sort_values(by='CR-MAE@7')

csv_path, npz_path = save_benchmark_artifacts(df_results, forecasts_dict, y_test_true, "statistical")
print(f">> Saved artifacts to:\n  - {csv_path}\n  - {npz_path}\n")

print("=== STATISTICAL BENCHMARK RESULTS (Ranked by CR-MAE@7) ===")
display(df_results[['model', 'group', 'strategy', 'MAE', 'RMSE', 'rain_day_MAE', 'CR-MAE@7', 'CR-Bias@7']])


>> Saved artifacts to:
  - data\processed\results\statistical_results.csv
  - data\processed\results\statistical_forecasts.npz

=== STATISTICAL BENCHMARK RESULTS (Ranked by CR-MAE@7) ===
               model        group  ... CR-MAE@7  CR-Bias@7
2              ARIMA  Statistical  ...  28.9757   -20.9845
4          AutoARIMA  Statistical  ...  28.9970   -21.1441
3         SARIMA_EDA  Statistical  ...  29.1896   -22.5888
1      SeasonalNaive     Baseline  ...  29.2470   -23.0190
5            AutoETS  Statistical  ...  29.6470   -26.0187
9              ADIDA  Statistical  ...  29.6470   -26.0193
10             IMAPA  Statistical  ...  29.6470   -26.0193
8                TSB  Statistical  ...  29.6542   -26.0733
6   CrostonOptimized  Statistical  ...  29.7726   -26.9608
7         CrostonSBA  Statistical  ...  29.7994   -27.1617
0              Naive     Baseline  ...  30.2570   -29.5090

[11 rows x 8 columns]
